<a href="https://colab.research.google.com/github/madiar-ops/AIpython/blob/main/neurons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Нейронные сети — MLPClassifier на датасете digits
# Google Colab Notebook — вставлять блоками по ячейкам
# ============================================================


# ════════════════════════════════════════════════════════════
# ЯЧЕЙКА 1 (Markdown)
# ════════════════════════════════════════════════════════════
"""
# 🧠 Нейронные сети: MLPClassifier на датасете digits

**Цель:** изучить работу многослойного персептрона (MLP),
сравнить архитектуры и проанализировать ошибки модели.

---
"""

# ════════════════════════════════════════════════════════════
# ЯЧЕЙКА 2 (Markdown)
# ════════════════════════════════════════════════════════════
"""
## Импорт библиотек
"""

# ════════════════════════════════════════════════════════════
# ЯЧЕЙКА 3 (Code) — Импорты
# ════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

digits = load_digits()

print(" Датасет load_digits загружен")
print(f"   digits.data.shape   = {digits.data.shape}    "
      f"← {digits.data.shape[0]} примеров, {digits.data.shape[1]} признаков (8×8 пикселей)")
print(f"   digits.images.shape = {digits.images.shape}  "
      f"← те же данные в формате (N, 8, 8)")
print(f"   digits.target.shape = {digits.target.shape}   "
      f"← метки классов от 0 до 9")
print(f"\nУникальные классы: {np.unique(digits.target)}")
print(f"Примеров в каждом классе:")
for cls in np.unique(digits.target):
    print(f"   Цифра {cls}: {np.sum(digits.target == cls)} примеров")



fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Первые 10 изображений датасета digits", fontsize=14, fontweight="bold")

for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap="gray_r")
    ax.set_title(f"Метка: {digits.target[i]}", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()



# Разделение 80% / 20%
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target,
    test_size=0.2,
    random_state=42,
    stratify=digits.target
)

print(f"Обучающая выборка: {X_train.shape[0]} примеров")
print(f"Тестовая выборка:  {X_test.shape[0]} примеров")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\n StandardScaler применён")
print(f"   Среднее (первые 5 признаков): {scaler.mean_[:5].round(2)}")
print(f"   Std    (первые 5 признаков): {scaler.scale_[:5].round(2)}")

base_model = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation="relu",
    max_iter=500,
    random_state=42
)
base_model.fit(X_train_scaled, y_train)

base_predictions = base_model.predict(X_test_scaled)
base_accuracy    = accuracy_score(y_test, base_predictions)

print(f"\n{'='*50}")
print(f"  Базовая модель MLP(100,) — Accuracy: {base_accuracy:.4f} ({base_accuracy*100:.2f}%)")
print(f"{'='*50}")
print("\nClassification Report:")
print(classification_report(y_test, base_predictions))

configs_task2 = [
    {"hidden_layer_sizes": (32,),  "label": "MLP(32)"},
    {"hidden_layer_sizes": (64,),  "label": "MLP(64)"},
    {"hidden_layer_sizes": (128,), "label": "MLP(128)"},
]

task2_results = []

print(f"{'Модель':<12} {'Accuracy':>10}  {'F1-macro':>10}")
print("-" * 36)

for cfg in configs_task2:
    model = MLPClassifier(
        hidden_layer_sizes=cfg["hidden_layer_sizes"],
        activation="relu",
        max_iter=500,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    preds    = model.predict(X_test_scaled)
    acc      = accuracy_score(y_test, preds)
    report   = classification_report(y_test, preds, output_dict=True)
    f1_macro = report["macro avg"]["f1-score"]

    task2_results.append({
        "label":   cfg["label"],
        "model":   model,
        "preds":   preds,
        "acc":     acc,
        "f1":      f1_macro,
    })
    print(f"{cfg['label']:<12} {acc:>10.4f}  {f1_macro:>10.4f}")

labels2 = [r["label"] for r in task2_results]
accs2   = [r["acc"]   for r in task2_results]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels2, accs2, color=["#4e79a7", "#f28e2b", "#59a14f"],
              edgecolor="white", width=0.5)
ax.set_title("Точность модели при разном числе нейронов", fontweight="bold")
ax.set_ylabel("Accuracy")
ax.set_ylim(min(accs2) - 0.02, 1.01)
for bar, val in zip(bars, accs2):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

configs_task3 = [
    {"hidden_layer_sizes": (64,),     "label": "MLP(64,)    — 1 слой"},
    {"hidden_layer_sizes": (64, 32),  "label": "MLP(64, 32) — 2 слоя"},
]

task3_results = []

print(f"{'Модель':<28} {'Accuracy':>10}  {'F1-macro':>10}")
print("-" * 52)

for cfg in configs_task3:
    model = MLPClassifier(
        hidden_layer_sizes=cfg["hidden_layer_sizes"],
        activation="relu",
        max_iter=500,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    preds    = model.predict(X_test_scaled)
    acc      = accuracy_score(y_test, preds)
    report   = classification_report(y_test, preds, output_dict=True)
    f1_macro = report["macro avg"]["f1-score"]

    task3_results.append({
        "label": cfg["label"],
        "model": model,
        "preds": preds,
        "acc":   acc,
        "f1":    f1_macro,
    })
    print(f"{cfg['label']:<28} {acc:>10.4f}  {f1_macro:>10.4f}")

for r in task3_results:
    print(f"\n── {r['label']} ──")
    print(classification_report(y_test, r["preds"]))


all_results = task2_results + task3_results + [{
    "label": "MLP(100) base",
    "model": base_model,
    "preds": base_predictions,
    "acc":   base_accuracy,
}]

best = max(all_results, key=lambda r: r["acc"])
nn_predictions = best["preds"]

print(f" Лучшая модель: {best['label']}")
print(f"   Accuracy: {best['acc']:.4f} ({best['acc']*100:.2f}%)")



cm = confusion_matrix(y_test, nn_predictions)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True, fmt="d",
    cmap="Blues",
    xticklabels=range(10),
    yticklabels=range(10),
    linewidths=0.5,
    ax=ax
)
ax.set_title(f"Confusion Matrix — {best['label']}", fontsize=13, fontweight="bold")
ax.set_xlabel("Предсказанная цифра", fontsize=11)
ax.set_ylabel("Истинная цифра",       fontsize=11)
plt.tight_layout()
plt.show()

errors = []
for true_cls in range(10):
    for pred_cls in range(10):
        if true_cls != pred_cls and cm[true_cls, pred_cls] > 0:
            errors.append((cm[true_cls, pred_cls], true_cls, pred_cls))

errors.sort(reverse=True)

print("\n Топ-5 пар цифр, которые модель путает чаще всего:")
print(f"{'#':<4} {'Истинная':>10} {'Предсказана':>13} {'Ошибок':>8}")
print("-" * 38)
for rank, (count, true_cls, pred_cls) in enumerate(errors[:5], 1):
    print(f"{rank:<4} {true_cls:>10} {pred_cls:>13} {count:>8}")


wrong_indexes = np.where(y_test != nn_predictions)[0]

print(f"Всего ошибок на тестовой выборке: {len(wrong_indexes)} из {len(y_test)}")
print(f"Точность: {(1 - len(wrong_indexes)/len(y_test))*100:.2f}%\n")

n_show = min(10, len(wrong_indexes))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Изображения, где модель ошиблась", fontsize=13, fontweight="bold")

for i, ax in enumerate(axes.flat):
    if i < n_show:
        idx = wrong_indexes[i]
        ax.imshow(X_test[idx].reshape(8, 8), cmap="gray_r")
        ax.set_title(
            f"Верно: {y_test[idx]}\nПред: {nn_predictions[idx]}",
            fontsize=9,
            color="red" if y_test[idx] != nn_predictions[idx] else "green"
        )
    ax.axis("off")

plt.tight_layout()
plt.show()



top2_pairs = errors[:2]

for count, true_cls, pred_cls in top2_pairs:
    pair_idxs = np.where(
        (y_test == true_cls) & (nn_predictions == pred_cls)
    )[0]

    print(f"\n Цифра {true_cls} → предсказана как {pred_cls}  ({count} ошибок)")

    n_pair = min(5, len(pair_idxs))
    if n_pair == 0:
        print("   Примеры не найдены в тестовой выборке.")
        continue

    fig, axes = plt.subplots(1, n_pair, figsize=(3 * n_pair, 3))
    if n_pair == 1:
        axes = [axes]
    fig.suptitle(
        f"Модель приняла «{true_cls}» за «{pred_cls}»",
        fontsize=12, fontweight="bold"
    )
    for ax, idx in zip(axes, pair_idxs[:n_pair]):
        ax.imshow(X_test[idx].reshape(8, 8), cmap="gray_r")
        ax.set_title(f"True:{y_test[idx]}, Pred:{nn_predictions[idx]}",
                     fontsize=9, color="red")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

